# 01 · Data store + evaluation setup  (steps 1-3)
**Accelerator: None (CPU).** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


* builds the BSDS500 array store (agreement maps from the per-annotator ground truth) and uploads it to HF
* copies the code to HF so other accounts don't need the Kaggle code dataset
* **step 1 gate:** scores the released PEdger checkpoint in every inference mode; one of them should
  reproduce the published ODS within ±0.002 (.821/.840 BSDS+VOC or .813/.834 BSDS-only).
  Scoring takes ~4 h per mode on CPU, so this may need two or three runs of this notebook: each run
  continues where the previous one stopped (progress is kept on Hugging Face).

In [ ]:
# ==== SETTINGS (nothing here needs to change between Kaggle accounts) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # the ONE private Hugging Face repo shared by every account
ACCOUNT = ""                                  # optional label, e.g. your name; sessions are told apart automatically
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Once per Kaggle account, and for every imported notebook:
#   Add-ons -> Secrets -> HF_TOKEN = a WRITE token from the Hugging Face account that owns HF_REPO, ticked for this notebook
#   Settings -> Internet: On;  Settings -> Accelerator: as written in the title
# The preflight cell below checks all of this and stops with a plain-language fix if something is missing.

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
def sh(cmd, cwd=None):
    """Run a shell command and show its output; stop the notebook if it fails (a plain `!cmd` carries on)."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for raw in p.stdout:                       # progress bars (\r updates) are shown as their last state
        print(raw.decode(errors="replace").rstrip("\r\n").split("\r")[-1], flush=True)
    if p.wait():
        raise RuntimeError(f"this step failed (exit code {p.returncode}); its error is printed above:\n  {cmd}")
os.environ["PEDGER_SESSION_START"] = str(SESSION_START)   # long steps stop and save before Kaggle's 12-hour limit
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Secret HF_TOKEN is not available: Add-ons -> Secrets -> add HF_TOKEN and tick it for THIS "
                       "notebook, then run again.") from e
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25,<2.0", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
# Always start from the newest code: an interactive session can still hold an old copy from an earlier run.
os.chdir(W)
shutil.rmtree(REPO, ignore_errors=True)
shutil.rmtree(f"{W}/hf_code", ignore_errors=True)
for _m in [m for m in list(sys.modules) if m == "pedger_kd" or m.startswith("pedger_kd.")]:
    del sys.modules[_m]
attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
if attached:                                       # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
    shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
    CODE_SOURCE = "kaggle dataset"
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    CODE_SOURCE = "git"
else:                                              # the copy kept on Hugging Face
    from huggingface_hub import snapshot_download
    snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                      token=os.environ["HF_TOKEN"])
    shutil.copytree(f"{W}/hf_code/code", REPO)
    CODE_SOURCE = "huggingface"
_stamp = os.path.join(REPO, "CODE_STAMP.json")
print("code from:", CODE_SOURCE, "| version:", open(_stamp).read().split('"')[3] if os.path.exists(_stamp) else "unstamped")
os.chdir(REPO)
sys.path.insert(0, REPO)
sh("python tools/fetch_upstream.py")
# preflight: session type, GPUs, internet, Hugging Face account and repo access
if subprocess.run([sys.executable, "-m", "pedger_kd.preflight"]).returncode:
    raise RuntimeError("Preflight failed: fix the PROBLEM lines printed above, then run the notebook again.")

In [ ]:
!pip install -q -r requirements.txt
!python tools/install_pyedgeeval.py   # BSDS benchmark scorer; patched build for Python 3.13+, ~1-2 min

In [ ]:
# share the code with every account through Hugging Face: uploads ONLY if this copy is newer than the one
# already there (e.g. a newer attached code dataset); an older copy is never pushed over a newer one
!python -m pedger_kd.sync push-code

In [ ]:
# BSDS500 original release (per-annotator ground truth); ~70 MB
RAW_BSDS = "/kaggle/working/BSR/BSDS500/data"
if not os.path.exists(RAW_BSDS):
    !wget -q https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz -O /kaggle/working/bsr.tgz
    !tar -xzf /kaggle/working/bsr.tgz -C /kaggle/working && rm /kaggle/working/bsr.tgz
print(sorted(os.listdir(RAW_BSDS + "/images")))

In [ ]:
from pedger_kd.sync import open_remote
remote = open_remote(os.environ["PEDGER_REMOTE"])
if remote.exists("data/bsds/meta.json"):
    print("data/bsds already on HF, pulling it")
    remote.download("data/bsds", WS)
else:
    !python data/prepare_bsds.py --bsds-root {RAW_BSDS} --out {WS}/data/bsds
    !python -m pedger_kd.sync push-dir --local {WS}/data/bsds --to data/bsds
!cat {WS}/data/bsds/meta.json

In [ ]:
# step 3 visual check: image | annotator fraction a | label Y | trust alpha
!python data/visualize_samples.py --store {WS}/data/bsds --split train --n 10 --out {W}/checks/agreement.png
from IPython.display import Image, display; display(Image(f"{W}/checks/agreement.png"))

In [ ]:
# step 1 gate: score the released PEdger checkpoint (single-scale for every mode first; multi-scale if MS=True).
# Scoring is slow on CPU (~4 h per mode for the 200 test images). Progress is saved to Hugging Face every
# 10 minutes and the run stops before Kaggle's 12-hour limit: just run the notebook again to continue.
# Modes already scored (in any session or account) are skipped. The table at the end compares with the paper.
MODES = ["ns_m", "rec_m", "fuse_m"]
MS = False
_modes, _ms = " ".join(MODES), ("--ms" if MS else "")
!python eval/released_gate.py --store {WS}/data/bsds --raw-root {RAW_BSDS} --modes {_modes} {_ms} --out {W}/released --device cpu

In [ ]:
# step 9 part 1: numeric BatchNorm check on the released weights
!python eval/bn_check.py --numeric --ckpt third_party/PEdger/checkpoint_PEdger.pth --device cpu

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")